# PolySight Seg — reproducción del entrenamiento

Este notebook verifica código, dataset y splits, ejecuta un smoke y permite repetir el entrenamiento completo. Test permanece desactivado por defecto. Se esperan métricas comparables, no checkpoints idénticos byte por byte.

In [ ]:
from pathlib import Path
import hashlib
import inspect
import os
import shutil
import subprocess
import sys

REPOSITORY = 'https://github.com/christianbueno1/polysight-seg.git'
REPO_REF = '2bf2c5a874272ecd6ccd24b936af578f4e637c82'
WORKSPACE = Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_ROOT = WORKSPACE / 'polysight-seg'
DATASET_SHA256 = '4463011f991dcdc74ec56399788b1a93822593f17ed18a662bdeb7392ffcdd9a'
MANIFEST_SHA256 = '35ddd003e5ec95817761c2e4de40c1c4274fc7ec43f7690d8b30aedee7019fd4'
SPLITS_SHA256 = '85fe68a5b241f880a80d1476fdffcff88ae5b5e51c0adbe690cce023cbfe13f9'
RUN_FULL_EXPERIMENT = False
RUN_TEST_EVALUATION = False  # Protección explícita; no cambiar para reproducir training
print({'python': sys.version, 'repo_ref': REPO_REF, 'full_experiment': RUN_FULL_EXPERIMENT, 'test': RUN_TEST_EVALUATION})

In [ ]:
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', REPO_REF], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run(['bash', 'scripts/validate_local.sh'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_inference_components.py', '-v'], check=True)

In [ ]:
import torch
from polysight_seg.data.splits import generate_splits
from polysight_seg.models.factory import build_model
from polysight_seg.training.checkpointing import save_epoch_checkpoints

print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
print(inspect.getsource(build_model))
print(inspect.getsource(generate_splits))
print(inspect.signature(save_epoch_checkpoints))

## Dataset inmutable

Define `DATASET_DRIVE_PATH` o sube el ZIP registrado. Se verifica el hash antes de extraer y se regeneran manifest y splits desde cero.

In [ ]:
DATASET_DRIVE_PATH = ''  # Ejemplo: /content/drive/MyDrive/polysight/hyper-kvasir-segmented-images.zip
DATASET_LOCAL_PATH = ''  # Para Jupyter/Kaggle; tiene prioridad sobre upload
archive_path = WORKSPACE / 'hyper-kvasir-segmented-images.zip'
if DATASET_LOCAL_PATH:
    shutil.copy2(DATASET_LOCAL_PATH, archive_path)
elif DATASET_DRIVE_PATH:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy2(DATASET_DRIVE_PATH, archive_path)
else:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError('Define DATASET_LOCAL_PATH fuera de Colab') from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Sube exactamente el ZIP de Kvasir-SEG')
    _, content = next(iter(uploaded.items()))
    archive_path.write_bytes(content)
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
if digest != DATASET_SHA256:
    raise RuntimeError(f'SHA-256 inesperado para el dataset: {digest}')
print({'dataset_sha256': digest, 'size_bytes': archive_path.stat().st_size})

In [ ]:
commands = [
    [sys.executable, 'scripts/prepare_dataset.py', str(archive_path)],
    [sys.executable, 'scripts/validate_dataset.py'],
    [sys.executable, 'scripts/generate_manifest.py'],
    [sys.executable, 'scripts/generate_splits.py', '--seed', '20260817'],
    [sys.executable, 'scripts/validate_splits.py'],
]
for command in commands:
    subprocess.run(command, check=True)
for path, expected in [
    (Path('data/processed/kvasir-seg/manifest.csv'), MANIFEST_SHA256),
    (Path('data/processed/kvasir-seg/splits.csv'), SPLITS_SHA256),
]:
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Hash inesperado para {path}: {actual}')
    print(path, actual)

## Smoke y entrenamiento opcional

El smoke usa dos batches de train y validation. El entrenamiento completo solo se ejecuta si `RUN_FULL_EXPERIMENT=True` y existe CUDA. No se ejecuta evaluación de test.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('El smoke de entrenamiento requiere activar un runtime GPU')
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--max-train-batches', '2',
    '--max-validation-batches', '2',
], check=True)
print('smoke_training=ok')

In [ ]:
if RUN_TEST_EVALUATION:
    raise RuntimeError('Test está desactivado por diseño en este notebook')
if RUN_FULL_EXPERIMENT:
    subprocess.run([sys.executable, 'scripts/train.py'], check=True)
    print('full_training=completed')
else:
    print('full_training=skipped; cambia RUN_FULL_EXPERIMENT a True para repetirlo')

## Persistir artefactos

Ejecuta esta celda al terminar para copiar un archivo comprimido a Drive. No se incluyen datos originales.

In [ ]:
import tarfile
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    bundle = WORKSPACE / 'polysight-colab-run.tar.gz'
    with tarfile.open(bundle, 'w:gz') as archive:
        for relative in ('checkpoints', 'artifacts', 'mlflow.db'):
            path = PROJECT_ROOT / relative
            if path.exists():
                archive.add(path, arcname=relative)
    destination = Path('/content/drive/MyDrive/polysight-runs')
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(bundle, destination / bundle.name)
    print('artefactos_guardados=', destination / bundle.name)
else:
    print('persistencia=omitida')